In [13]:
import os
import sys
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd

sys.path.append(os.path.abspath(".."))
from function import ART_bias
from function import ART_downscale as ART_down

import warnings
warnings.filterwarnings('ignore')

## Export Correceted PARAMs for each seed

In [14]:
# product, time_reso = 'IMERG', '1dy'
# product, time_reso = 'CMORPH', '3h'
product, time_reso = 'CHIRPS', '1dy'

# product, time_reso = 'ERA5', '3h'
# product, time_reso = 'MSWEP', '3h'
# product, time_reso = 'GSMaP', '3h'

In [15]:
COMPUTERNAME = os.environ['COMPUTERNAME']
print(f'Computer: {COMPUTERNAME}')

if COMPUTERNAME == 'BR_DELL':
    dir_font = os.path.join('/','run')
else:
    dir_font = os.path.join('/')

dir_base = os.path.join('/','media','arturo','T9','Data','Italy')

veneto_dir = os.path.join(dir_font,'media','arturo','T9','Data','shapes','Europa','Italy')

obs_base = os.path.join(dir_font,'media','arturo','T9','Data','Italy','Rain_Gauges_QC')

dir_cal = os.path.join('/','media','arturo','T9','Data','Italy','Rain_Gauges_QC','CAL_VAL', 'Calibration')
dir_val = os.path.join('/','media','arturo','T9','Data','Italy','Rain_Gauges_QC','CAL_VAL', 'Validation')

Computer: UNIPD_DELL


In [16]:
if os.path.exists(veneto_dir):
    ITALY = gpd.read_file(os.path.join(veneto_dir,'Italy_clear.geojson'))
else:
    raise SystemExit(f"File not found: {veneto_dir}")

In [17]:
# seeds_list = [7, 19, 31, 53, 89, 127, 211, 307, 401, 509, 613, 727, 839, 947, 1051]

primos_50 = [2, 3, 5, 7, 11, 13, 17, 19, 23, 29, 31, 37, 41, 43, 47, 53, 59, 61, 67, 71,
            73, 79, 83, 89, 97, 101, 103, 107, 109, 113, 127, 131, 137, 139, 149, 151,
            157, 163, 167, 173, 179, 181, 191, 193, 197, 199, 211, 223, 227, 229]

np.random.seed(123)
aleatorios_50 = np.random.choice(range(1000, 10000), size=50, replace=False).tolist()

seeds_list = primos_50 + aleatorios_50

In [18]:
Tr = [5,  10,  20,  50, 100, 200]
Fi = 1 - 1/np.array(Tr)

for seed in seeds_list:

    data_dir = os.path.join(dir_base, 'Satellite','5_DOWN', f'ITALY_DOWN_{product}_{time_reso}_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson.nc')
    DATA = xr.open_dataset(data_dir)

    lons, lats = DATA.lon.values, DATA.lat.values
    lon2d, lat2d = np.meshgrid(lons, lats)
    years = DATA.year.values

    METADATA_CAL = pd.read_csv(os.path.join(dir_cal, f'METADATA_CAL_seed{seed}.csv'))

    for nn in range(len(METADATA_CAL)):

        name = METADATA_CAL['File'].values[nn]
        ISO = METADATA_CAL['ISO'].values[nn]
        lat_obs = METADATA_CAL['Lat'].values[nn]
        lon_obs = METADATA_CAL['Lon'].values[nn]

        dir_station = os.path.join(dir_base,'Rain_Gauges_QC','Weibull','1dy',ISO,name)
        DF_OBS = pd.read_csv(dir_station)
        DF_OBS = DF_OBS.drop(columns='NaN')

        PREC_DOWN = DATA.sel(lat=lat_obs, lon=lon_obs, method='nearest')
        
        DF_SAT = pd.DataFrame({
                            'Year':PREC_DOWN.year.values, 
                            'Ns':PREC_DOWN.NYs.values, 'Cs':PREC_DOWN.CYs.values, 'Ws':PREC_DOWN.WYs.values,
                            'Nd':PREC_DOWN.NYd.values, 'Cd':PREC_DOWN.CYd.values, 'Wd':PREC_DOWN.WYd.values})

        merged_df = pd.merge(DF_SAT, DF_OBS, on='Year', how='left')

        merged_df_clean = merged_df.dropna(subset=['N'])

    NYs_corrected = ART_bias.linear_bc_through_origin(merged_df_clean.N.values,merged_df_clean.Ns.values,DATA.NYs.values)
    CYs_corrected = ART_bias.linear_bc_through_origin(merged_df_clean.C.values,merged_df_clean.Cs.values,DATA.CYs.values)
    WYs_corrected = ART_bias.linear_bc_through_origin(merged_df_clean.W.values,merged_df_clean.Ws.values,DATA.WYs.values)

    Mevs_corrected = ART_down.pre_quantiles_array(
                                    NYs_corrected,
                                    CYs_corrected, 
                                    WYs_corrected,
                                    Tr, lats, lons, 1)

    NYd_corrected = ART_bias.linear_bc_through_origin(merged_df_clean.N.values,merged_df_clean.Nd.values,DATA.NYd.values)
    CYd_corrected = ART_bias.linear_bc_through_origin(merged_df_clean.C.values,merged_df_clean.Cd.values,DATA.CYd.values)
    WYd_corrected = ART_bias.linear_bc_through_origin(merged_df_clean.W.values,merged_df_clean.Wd.values,DATA.WYd.values)

    Mevd_corrected = ART_down.pre_quantiles_array(
                                    NYd_corrected,
                                    CYd_corrected, 
                                    WYd_corrected,
                                    Tr, lats, lons, 1)

    SAT_corrected = xr.Dataset(
                            data_vars={
                                "NYs": (("year","lat","lon"), NYs_corrected),
                                "CYs": (("year","lat","lon"), CYs_corrected),
                                "WYs": (("year","lat","lon"), WYs_corrected),
                                "NYd": (("year","lat","lon"), NYd_corrected),
                                "CYd": (("year","lat","lon"), CYd_corrected),
                                "WYd": (("year","lat","lon"), WYd_corrected),
                                "Mev_s":(("Tr","lat","lon"), Mevs_corrected),
                                "Mev_d":(("Tr","lat","lon"), Mevd_corrected),
                                },
                            coords={
                                'year': years, 
                                'Tr':Tr,
                                'lat': lats, 
                                'lon': lons
                                },
                            attrs=dict(description=f"{product} Weibull parameters and MEV corrected applying Linear-regression Through Origin  (LTO) method using seed {seed} and 70% of stations as calibration",))

    dir_out = os.path.join('/','media','arturo','T9','Data','Italy','Satellite','6_DOWN_BCorrected','PARAM','LTO')
    PRE_out = os.path.join(os.path.join(dir_out, data_dir.split('/')[-1].replace('_pearson',f'_pearson_{str(seed).zfill(4)}')))
    print(f'Exportin as: {PRE_out}')
    SAT_corrected.to_netcdf(PRE_out)

Exportin as: /media/arturo/T9/Data/Italy/Satellite/6_DOWN_BCorrected/PARAM/LTO/ITALY_DOWN_CHIRPS_1dy_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_0002.nc
Exportin as: /media/arturo/T9/Data/Italy/Satellite/6_DOWN_BCorrected/PARAM/LTO/ITALY_DOWN_CHIRPS_1dy_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_0003.nc
Exportin as: /media/arturo/T9/Data/Italy/Satellite/6_DOWN_BCorrected/PARAM/LTO/ITALY_DOWN_CHIRPS_1dy_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_0005.nc
Exportin as: /media/arturo/T9/Data/Italy/Satellite/6_DOWN_BCorrected/PARAM/LTO/ITALY_DOWN_CHIRPS_1dy_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_0007.nc
Exportin as: /media/arturo/T9/Data/Italy/Satellite/6_DOWN_BCorrected/PARAM/LTO/ITALY_DOWN_CHIRPS_1dy_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_0011.nc
Exportin as: /media/arturo/T9/Data/Italy/Satellite/6_DOWN_BCorrected/PARAM/LTO/ITALY_DOWN_CHIRPS_1dy_2002_2023_npix_2_thr_1_acf_mar_genetic_pearson_0013.nc
Exportin as: /media/arturo/T9/Data/Italy/Satellite/6_DOWN_BCorre

KeyboardInterrupt: 